## 05-2 교차 검증과 그리드 서치
---

전에는 와인 분류 문제를 해결하기 위해 로지스틱 회귀를 사용했지만, 모델의 구조를 직관적으로 이해하기 어렵다는 한계가 있었다.\
이에 따라, 보다 해석이 쉬운 결정 트리를 사용하여 문제를 해결해 보았다.

<br>

지금까지 우리는 훈련 세트에서 모델을 훈련하고 테스트 세트에서 모델을 평가했다.\
그런 다음 테스트 세트에서의 점수를 보고 일반화 성능을 가늠해 본 것이었다.\
근데, 이렇게 테스트 세트를 사용해 자꾸 성능을 확인하다 보면 테스트 세트에 점차 맞춰지는 것 아닐까?
<br>

일반화 성능을 올바르게 예측하고 싶다면 가능한 사용하지 않고 마지막에 딱 한 번만 사용하는 것이 좋을 것이다.\
이런 것들을 해결할 수 있을까 살펴보자.

---

### 검증 세트

테스트 세트를 사용하지 않으면 모델의 과대적합, 과소적합을 확인할 수 없어진다.\
따라서 우리는 이를 확인하기 위해서 훈련 세트를 또 나눈다. &rArr; 이를 **검증 세트(Validation set)** 이라고 한다.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

wine = pd.read_csv("./data/wine_csv_data.csv")
wine.head()

data = wine[["alcohol", "sugar", "pH"]]
target = wine["class"]

train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42
)

# train_input 과 train_target을 다시 분할하여서 얻는다.
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42
)

print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [4]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)

print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


이런 방식으로 `val_input`과 `val_target`을 만들어 평가하면 된다.\
현재는 모델이 훈련 세트에 과대적합 되어 있음을 확인할 수 있다.

---

### 교차 검증

검증 세트를 만드느라 훈련 세트가 줄어버렸다...\

- 보통은 많은 데이터를 훈련에 사용할 수록 좋은 모델이 만들어지게 된다.
- 그렇다고 해서 너무 조금 떼어 놓으면 검증 점수가 더 불안정해 질 수 있다.

이럴 때 **교차 검증(Cross Validation)** 을 활용하면 안정적인 검증 점수를 얻고 훈련에 더 많은 데이터를 활용할 수도 있다.

<br>

교차 검증은 검증 세트를 떼어 내어 평가하는 과정을 여러 번 반복한다.

![cross_validation](./data/image/cross_validation.png)

사이킷런 안에는 `cross_validate()`라는 교차 검증 함수가 존재한다.
- 평가할 객체를 첫 번째 매개변수로 전달한다.
- 앞에서 처럼 검증 세트를 떼어 내지 않고 훈련 세트 전체를 함수에 전달한다.


In [7]:
from sklearn.model_selection import cross_validate
import numpy as np

scores = cross_validate(dt, train_input, train_target)
print(scores)

# 교차 검증의 최종 점수는 test_score에 담긴 점수를 평균하여 얻을 수 있다.
print(np.mean(scores["test_score"]))

{'fit_time': array([0.01042128, 0.01100397, 0.00843811, 0.01701617, 0.01084471]), 'score_time': array([0.00307846, 0.00516844, 0.        , 0.00153565, 0.        ]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}
0.855300214703487


한 가지 주의할 점은 `cross_validate()`는 훈련 세트를 섞어 폴드를 나누지 않는다.\
앞서서는 우리가 `train_test_split()` 함수로 전체 데이터를 섞은 다음에 훈련 세트를 준비했기 때문에 섞을 필요가 없었다.

교차 검증을 할 때 훈련 세트를 섞기 위해서는 분할기(splitter)를 사용해야 한다.
- 기본적으로 회귀 모델일 경우에는 `KFold` 분할기를 사용하고
- 분류 모델일 경우에는 `StratifiedKFold`를 사용한다.

In [13]:
# 해당 코드는 위의 코드와 동일한 코드이다.
from sklearn.model_selection import StratifiedKFold

scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores["test_score"]))

# 만약 훈련 세트를 섞은 후 10-폴드 교차 검증을 수행하고 싶다면
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores["test_score"]))

0.855300214703487
0.8574181117533719


이제는 결정 트리의 매개변수 값을 바꿔가면서 가장 좋은 성능이 나오는 모델을 찾으면 된다.\
그 과정에서 테스트 세트를 사용하지 않고 교차 검증을 통해서 좋은 모델을 고를 수 있는 것이다.

---

### 하이퍼파라미터 튜닝

머신러닝 모델이 학습하는 파라미터를 "모델 파라미터",\
학습할 수 없어서 사용자가 지정해야만 하는 파라미터를 "하이퍼 파라미터"라고 한다.

이러한 하이퍼 파라미터의 튜닝 작업은 어떻게 이루어질까?
- 먼저 라이브러리가 제공하는 기본값을 활용하여 훈련을 한다.
- 그런 다음 검증 세트나 교차 검증을 통해서 매개변수를 조금씩 바꾸면서 진행된다.

<br>

예를 들어, `max_depth` 값을 최적의 값으로 고정시키고 `min_samples_split`을 바꿔가면서 최적의 값을 찾았다고 해보자.\
그런 다음 이제 또 다른 매개변수의 최적의 값을 찾아도 되는 걸까?\
아쉽지만, 한 매개변수의 값이 바뀌면 또 다른 매개변수의 최적의 값도 바뀌게 되기 때문에, 두 매개변수를 동시에 바꿔가며 찾아야 하는 것이다.

<br>

이러한 문제는 매개변수가 많아질수록 더 복잡해진다. 파이전의 `for`반복문을 이용하여 해결할 수 있지도 않을까 싶지만,\
사이킷런에서는 이미 제공하고 있는 도구가 존재한다. &rArr; **그리드 서치(Grid Search)**

In [15]:
from sklearn.model_selection import GridSearchCV

# GridSearchCV는 하이퍼파라미터 탐색과 교차 검증을 한 번에 수행한다. (cross_validate()를 호출할 필요가 없다.)
# 기본 매개변수를 사용한 결정트리에서 min_impurity_decrease 매개변수의 최적값을 찾아보자.

params = {"min_impurity_decrease": [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

# GridSearchCV에서는 많은 모델을 훈련하기 때문에 
# n_jobs 매개변수에서 병렬 실행에 사용할 CPU 코어 수를 지정하는 것이 좋다. (-1은 시스템에 있는 모든 코어 사용)
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'min_impurity_decrease': [0.0001, 0.0002, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fo

교차 검증에서 최적의 하이퍼파라미터를 찾으면 전체 훈련 세트로 모델을 다시 만들어야 한다.\

근데 사이킷런의 그리드 서치는 훈련이 끝나면 교차 검증 점수가 가장 높은 모델의 매개변수 조합으로\
전체 훈련 세트에서 자동으로 다시 모델을 훈련한다.\
&rArr; 해당 모델은 gs객체의 `best_estimator_`속성에 저장되어 있다.\
&rArr; 최적의 매개변수는 `best_params_`속성에 저장되어 있다.\
&rArr; 각 매개변수에서 수행한 교차 검증의 평균 점수는 `cv_results_`속성 안에서 확인해볼 수 있다.

In [ ]:
dt = gs.best_estimator_

print(dt.score(train_input, train_target))
print(gs.best_params_)

print(gs.cv_results_["mean_test_score"]) # 이 중 첫번재 값이 가장 크므로 해당 값의 매개변수 가져와 보자

# gs객체의 "best_index_"속성을 통해 가장 높은 값의 인덱스를 얻을 수 있다.
print(gs.cv_results_["params"][gs.best_index_])

0.9615162593804117
{'min_impurity_decrease': 0.0001}
[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]
{'min_impurity_decrease': 0.0001}


조금 더 복잡하게 가보자.

In [27]:
# arange()와 range() 함수의 기능은 동일하지만,
# arange()는 실수에 까지 사용 가능하고, range()는 정수에만 사용 가능하다.
parmas = {
    "min_impurity_decrease": np.arange(0.0001, 0.001, 0.0001),
    "max_depth": range(5, 20, 1),
    "min_samples_split": range(2, 100, 10),
          }

gs = GridSearchCV(DecisionTreeClassifier(random_state=42), parmas, n_jobs=-1)
gs.fit(train_input, train_target)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': range(5, 20), 'min_impurity_decrease': array([0.0001... 0.0009]), 'min_samples_split': range(2, 100, 10)}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter 

In [ ]:
print(gs.best_score_)
print(gs.best_params_)

0.8683865773302731
{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}


---

### 랜덤 서치

그런데 매개변수의 값이 수치일 때, 값의 범위나 간격을 미리 정하기 어려울 수 있다.\
또 너무 많은 매개변수 조건이 존재해 그리드 서치 수행 시간이 오래 걸릴 수 있다.\
&rArr; 이럴 때 존재하는 것이 바로 **랜덤 서치(Random Search)** 이다.

<br> 

랜덤 서치에는 매개변수 값의 목록을 전달하는 것이 아니라,\
매개변수를 샘플링할 수 있는 "확률 분포 객체"를 전달한다.

In [38]:
from scipy.stats import uniform, randint

# stats 서브 패키지에 있는 uniform과 randint 클래스는
# 주어진 범위에서 고르게 값을 뽑는다. => radint는 정수값, uniform은 실수값을 뽑는다.

rgen = randint(0, 10) # 0~10 범위의 갖는 randint 객체
rgen.rvs(10)

np.unique(rgen.rvs(1000), return_counts=True)

ugen = uniform(0, 1)
ugen.rvs(10)

array([0.56661996, 0.81269596, 0.55385063, 0.96604441, 0.31894765,
       0.68182787, 0.28593586, 0.49287085, 0.93913075, 0.75229894])

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

parmas = {
    "min_impurity_decrease": uniform(0.0001, 0.001),
    "max_depth": randint(20, 50),
    "min_samples_split": randint(2, 25),
    "min_samples_leaf": randint(1, 25),
          }

rs = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42), params, n_iter=100, n_jobs=-1, random_state=42)
   
rs.fit(train_input, train_target)

print(rs.best_params_)

{'min_impurity_decrease': 0.0001}


c:\Users\Winter\Desktop\Study\TIL\AI\MLDL(26-03-17)\venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 5 is smaller than n_iter=100. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [47]:
print(rs.best_params_)

{'min_impurity_decrease': 0.0001}
